# DragonScope GPU training

1. In Colab, choose **Runtime > Change runtime type > T4 GPU**.
2. Run all cells. The upload cell asks for `dragonscope_colab.zip`, made on your laptop with:
   `git archive -o dragonscope_colab.zip HEAD`
3. If the session dies, run all again. Training resumes from the checkpoint in Google Drive.
4. Download `best_model.pt` from `MyDrive/dragonscope-runs/gpu-main/` and commit it to `models/`.

In [ ]:
import os, shutil, zipfile
from google.colab import drive, files

drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/dragonscope-runs/gpu-main'
os.makedirs(RUN_DIR, exist_ok=True)

shutil.rmtree('/content/dragonscope', ignore_errors=True)
name = list(files.upload())[0]
with zipfile.ZipFile(name) as archive:
    archive.extractall('/content/dragonscope')
os.chdir('/content/dragonscope')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m dragonscope.data --out data
!python -m pip install -q pytest && python -m pytest -q tests/test_engine.py

In [ ]:
!python -m dragonscope.train --data data --out "{RUN_DIR}" --iters 30000 --batch 32 --eval-every 500 --save-every-seconds 300